In [1]:
import os
import pandas as pd

In [2]:
import requests
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FMP_API_KEY")

In [4]:
# 2. Configure the endpoint and parameters
# Note: FMP requires the apikey as a query parameter
url = "https://financialmodelingprep.com/stable/stock-list"
params = {
    "apikey": api_key
}

# 3. Make the request
try:
    response = requests.get(url, params=params)
    response.raise_for_status() # Check for HTTP errors
    symbols_data = response.json()

    print(f"Total symbols retrieved: {len(symbols_data)}")

    # 4. Preview the data
    if symbols_data:
        print("\nSample Data:")
        # Display the first 5 results in a clean format using a DataFrame
        df = pd.DataFrame(symbols_data)
        print(df.head())

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

Total symbols retrieved: 89838

Sample Data:
     symbol                                  companyName
0     OGFGF                        Origin Energy Limited
1    USMV.L  Ossiam US Minimum Variance ESG NR UCITS ETF
2     MSSAR       Metal Sky Star Acquisition Corporation
3  AN3PL.AX  Australia and New Zealand Banking Group Ltd
4       RSF     RiverNorth Capital and Income Fund, Inc.


In [6]:
df.to_csv("data/stock-list.csv", index=False)

In [7]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv


# Mapping of filename to their respective endpoints
reference_endpoints = {
    "available-exchanges": "https://financialmodelingprep.com/stable/available-exchanges",
    "available-sectors": "https://financialmodelingprep.com/stable/available-sectors",
    "available-industries": "https://financialmodelingprep.com/stable/available-industries",
    "available-countries": "https://financialmodelingprep.com/stable/available-countries"
}

for name, url in reference_endpoints.items():
    response = requests.get(url, params={"apikey": api_key})
    if response.status_code == 200:
        data = response.json()
        # Some return a list of strings, some a list of dicts. 
        # DataFrame handles both well.
        temp_df = pd.DataFrame(data) 
        temp_df.to_csv(f"data/{name}.csv", index=False)
        print(f"Saved: data/{name}.csv ({len(data)} items)")
    else:
        print(f"Failed to fetch {name}: {response.status_code}")

Saved: data/available-exchanges.csv (64 items)
Saved: data/available-sectors.csv (11 items)
Saved: data/available-industries.csv (159 items)
Saved: data/available-countries.csv (117 items)


In [9]:
import os
import time
import random
import requests
import pandas as pd
import io
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
api_key = os.getenv("FMP_API_KEY")

def fetch_bulk_data(url, params, max_retries=5):
    for n in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            
            if response.status_code == 200:
                # 1. Check if the response is empty
                if not response.text.strip():
                    return None 

                # 2. Check Content-Type to decide how to parse
                content_type = response.headers.get('Content-Type', '').lower()
                
                if 'csv' in content_type or not response.text.strip().startswith('['):
                    # Treat as CSV
                    return pd.read_csv(io.StringIO(response.text))
                else:
                    # Treat as JSON
                    try:
                        return pd.DataFrame(response.json())
                    except ValueError:
                        # Handle the "Extra Data" case (multiple JSON objects)
                        # This trick splits concatenated JSON objects if needed
                        import json
                        import re
                        # Basic fix for concatenated JSON: [{...}][{...}] -> [{...}, {...}]
                        cleaned_json = response.text.replace('][', '],[')
                        if not cleaned_json.startswith('['): 
                            cleaned_json = f"[{cleaned_json}]"
                        return pd.DataFrame(json.loads(cleaned_json))

            elif response.status_code == 429 or 500 <= response.status_code < 600:
                wait = (2 ** (n + 1)) + random.uniform(0, 1)
                print(f"Status {response.status_code}. Retrying in {wait:.2f}s...")
                time.sleep(wait)
            else:
                print(f"Error {response.status_code}: {response.text[:100]}")
                return None
                
        except Exception as e:
            wait = (2 ** (n + 1)) + random.uniform(0, 1)
            print(f"Request failed: {e}. Retrying in {wait:.2f}s...")
            time.sleep(wait)
    return None

# --- Main Loop ---
all_dfs = []
part = 0
bulk_url = "https://financialmodelingprep.com/stable/profile-bulk"

print("Starting bulk download...")

while True:
    print(f"Fetching part {part}...")
    params = {"part": part, "apikey": api_key}
    
    df_part = fetch_bulk_data(bulk_url, params)
    
    # If None or empty, we've likely hit the end or a fatal error
    if df_part is None or df_part.empty:
        print("No more data or error reached.")
        break
        
    all_dfs.append(df_part)
    print(f"  Part {part} loaded: {len(df_part)} rows.")
    
    part += 1
    time.sleep(0.5) # Be kind to the API

# Concatenate all parts into one master DataFrame
if all_dfs:
    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.to_csv("data/company-profiles-bulk.csv", index=False)
    print(f"\nSuccess! Total records: {len(full_df)}")
    print("Saved to data/company-profiles-bulk.csv")

Starting bulk download...
Fetching part 0...


/tmp/ipykernel_69091/3141079093.py:27: DtypeWarning: Columns (0: fullTimeEmployees) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(response.text))


  Part 0 loaded: 22375 rows.
Fetching part 1...
  Part 1 loaded: 22375 rows.
Fetching part 2...
  Part 2 loaded: 22371 rows.
Fetching part 3...
  Part 3 loaded: 22321 rows.
Fetching part 4...
Error 400: Query Error: Invalid or missing query parameter - part
No more data or error reached.

Success! Total records: 89442
Saved to data/company-profiles-bulk.csv


In [58]:
full_df = pd.read_csv('data/company-profiles-bulk.csv')
full_df

/tmp/ipykernel_69091/3399938694.py:1: DtypeWarning: Columns (0: fullTimeEmployees) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv('data/company-profiles-bulk.csv')


,symbol,price,marketCap,beta,lastDividend,range,change,changePercentage,volume,averageVolume,companyName,currency,cik,isin,cusip,exchangeFullName,exchange,industry,website,description,ceo,sector,country,fullTimeEmployees,phone,address,city,state,zip,image,ipoDate,defaultImage,isEtf,isActivelyTrading,isAdr,isFund
0,NVDA,228.19,"5,545,017,000,000.00",2.24,0.04,129.16-236.54,-7.55,-3.20,"92,659,853.33","170,298,113.00",NVIDIA Corporation,USD,"1,045,810.00",US67066G1040,67066G104,NASDAQ Global Select,NASDAQ,Semiconductors,https://www.nvidia.com,"NVIDIA Corporation provides graphics, and comp...",Jen-Hsun Huang,Technology,US,"36,000.00",408 486 2000,2788 San Tomas Expressway,Santa Clara,CA,95051,https://images.financialmodelingprep.com/symbo...,1999-01-22,False,False,True,False,False
1,KVYO,14.63,"4,378,463,723.00",0.79,0.00,13.53-36.76,0.29,2.02,"2,833,205.00","5,295,504.00","Klaviyo, Inc.",USD,"1,835,830.00",US49845K1016,49845K101,New York Stock Exchange,NYSE,Software - Infrastructure,https://www.klaviyo.com,"Klaviyo, Inc., a technology company that provi...",Andrew Bialecki,Technology,US,"2,316.00",(617) 213-1788,Floor 6,Boston,MA,02111,https://images.financialmodelingprep.com/symbo...,2023-09-20,False,False,True,False,False
2,SATS,137.35,"39,586,294,286.00",0.96,0.00,14.9-139.54,2.24,1.66,"2,465,326.93","6,007,778.00",EchoStar Corporation,USD,"1,415,404.00",US2787681061,278768106,NASDAQ Global Select,NASDAQ,Communication Equipment,https://www.echostar.com,"EchoStar Corporation, together with its subsid...",Hamid Akhavan-Malayeri,Technology,US,"13,700.00",303 706 4000,100 Inverness Terrace East,Englewood,CO,80112-5308,https://images.financialmodelingprep.com/symbo...,2008-01-02,False,False,True,False,False
3,SHC,15.41,"4,395,820,696.00",1.82,0.00,10.795-19.85,-0.08,-0.48,"2,577,312.09","3,416,172.00",Sotera Health Company,USD,"1,822,479.00",US83601L1026,83601L102,NASDAQ Global Select,NASDAQ,Medical - Diagnostics & Research,https://soterahealth.com,"Sotera Health Company provides sterilization, ...",Michael Petras Jr.,Healthcare,US,"3,000.00",440 262 1410,9100 South Hills Boulevard,Broadview Heights,OH,44147,https://images.financialmodelingprep.com/symbo...,2020-11-20,False,False,True,False,False
4,STRC,99.29,"29,495,990,704.00",3.60,11.50,90.52-100.418,-0.71,-0.71,"3,666,699.07","2,985,974.00",MicroStrategy Incorporated Variable Rate Serie...,USD,"1,050,446.00",US5949728530,594972408,NASDAQ Global Select,NASDAQ,Software - Application,http://www.strategy.com,"MicroStrategy, Inc. engages in the provision o...",Phong Q. Le,Technology,US,"1,534.00",703 848 8600,1850 Towers Crescent Plaza,Vienna,VA,22182,https://images.financialmodelingprep.com/symbo...,2025-07-30,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89437,BRAS.CN,0.01,"250,432.00",0.54,0.00,0.005-0.06,-0.01,-50.00,"51,000.00","7,438.00",Nordique Resources Inc.,CAD,NaN,NaN,NaN,Canadian Securities Exchange,CNQ,Gold,http://www.brascangold.com,"Nordique Resources, Inc. is a mineral explorat...",Damion Carruel,Basic Materials,CA,NaN,604 812 1747,409 Granville Street,Vancouver,BC,V6C 1T2,https://images.financialmodelingprep.com/symbo...,2021-10-14,True,False,False,False,False
89438,G3E.L,0.00,156.00,0.50,0.00,1.0E-4-0.281,-28.10,-100.00,"250,000.00",0.00,G3 Exploration Limited,GBp,NaN,NaN,NaN,London Stock Exchange,LSE,Oil & Gas Exploration & Production,https://www.g3-ex.com,"G3 Exploration Limited, an investment holding ...",Randeep Singh Grewal,Energy,CN,NaN,86 371 6013 3388,Landmark Plaza,Zhengzhou,NaN,450000,https://images.financialmodelingprep.com/symbo...,2006-08-17,True,False,False,False,False
89439,T1EU.F,40.24,0.00,0.01,0.00,40.244-42.595,-0.03,-0.07,295.00,31.00,Invesco US Treasury Bond 0-1 Year UCITS ETF,EUR,NaN,IE00BLCH1X54,NaN,Frankfurt Stock Exchange,FSX,Asset Management - Bonds,NaN,NaN,NaN,Financial Services,IE,NaN,NaN,NaN,NaN,NaN,NaN,htt

In [11]:
pd.set_option('display.max_columns', None)
full_df

,symbol,price,marketCap,beta,lastDividend,range,change,changePercentage,volume,averageVolume,companyName,currency,cik,isin,cusip,exchangeFullName,exchange,industry,website,description,ceo,sector,country,fullTimeEmployees,phone,address,city,state,zip,image,ipoDate,defaultImage,isEtf,isActivelyTrading,isAdr,isFund
0,NVDA,228.1900,5.545017e+12,2.244000,0.04,129.16-236.54,-7.5500,-3.20268,9.265985e+07,170298113.0,NVIDIA Corporation,USD,1045810.0,US67066G1040,67066G104,NASDAQ Global Select,NASDAQ,Semiconductors,https://www.nvidia.com,"NVIDIA Corporation provides graphics, and comp...",Jen-Hsun Huang,Technology,US,36000.0,408 486 2000,2788 San Tomas Expressway,Santa Clara,CA,95051,https://images.financialmodelingprep.com/symbo...,1999-01-22,False,False,True,False,False
1,KVYO,14.6300,4.378464e+09,0.791000,0.00,13.53-36.76,0.2900,2.02232,2.833205e+06,5295504.0,"Klaviyo, Inc.",USD,1835830.0,US49845K1016,49845K101,New York Stock Exchange,NYSE,Software - Infrastructure,https://www.klaviyo.com,"Klaviyo, Inc., a technology company that provi...",Andrew Bialecki,Technology,US,2316.0,(617) 213-1788,Floor 6,Boston,MA,02111,https://images.financialmodelingprep.com/symbo...,2023-09-20,False,False,True,False,False
2,SATS,137.3499,3.958629e+10,0.960000,0.00,14.9-139.54,2.2399,1.65783,2.465327e+06,6007778.0,EchoStar Corporation,USD,1415404.0,US2787681061,278768106,NASDAQ Global Select,NASDAQ,Communication Equipment,https://www.echostar.com,"EchoStar Corporation, together with its subsid...",Hamid Akhavan-Malayeri,Technology,US,13700.0,303 706 4000,100 Inverness Terrace East,Englewood,CO,80112-5308,https://images.financialmodelingprep.com/symbo...,2008-01-02,False,False,True,False,False
3,SHC,15.4149,4.395821e+09,1.823000,0.00,10.795-19.85,-0.0751,-0.48483,2.577312e+06,3416172.0,Sotera Health Company,USD,1822479.0,US83601L1026,83601L102,NASDAQ Global Select,NASDAQ,Medical - Diagnostics & Research,https://soterahealth.com,"Sotera Health Company provides sterilization, ...",Michael Petras Jr.,Healthcare,US,3000.0,440 262 1410,9100 South Hills Boulevard,Broadview Heights,OH,44147,https://images.financialmodelingprep.com/symbo...,2020-11-20,False,False,True,False,False
4,STRC,99.2888,2.949599e+10,3.595000,11.50,90.52-100.418,-0.7112,-0.71120,3.666699e+06,2985974.0,MicroStrategy Incorporated Variable Rate Serie...,USD,1050446.0,US5949728530,594972408,NASDAQ Global Select,NASDAQ,Software - Application,http://www.strategy.com,"MicroStrategy, Inc. engages in the provision o...",Phong Q. Le,Technology,US,1534.0,703 848 8600,1850 Towers Crescent Plaza,Vienna,VA,22182,https://images.financialmodelingprep.com/symbo...,2025-07-30,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89437,BRAS.CN,0.0050,2.504320e+05,0.537591,0.00,0.005-0.06,-0.0050,-50.00000,5.100000e+04,7438.0,Nordique Resources Inc.,CAD,NaN,NaN,NaN,Canadian Securities Exchange,CNQ,Gold,http://www.brascangold.com,"Nordique Resources, Inc. is a mineral explorat...",Damion Carruel,Basic Materials,CA,NaN,604 812 1747,409 Granville Street,Vancouver,BC,V6C 1T2,https://images.financialmodelingprep.com/symbo...,2021-10-14,True,False,False,False,False
89438,G3E.L,0.0001,1.560000e+02,0.497708,0.00,1.0E-4-0.281,-28.0999,-99.99960,2.500000e+05,0.0,G3 Exploration Limited,GBp,NaN,NaN,NaN,London Stock Exchange,LSE,Oil & Gas Exploration & Production,https://www.g3-ex.com,"G3 Exploration Limited, an investment holding ...",Randeep Singh Grewal,Energy,CN,NaN,86 371 6013 3388,Landmark Plaza,Zhengzhou,NaN,450000,https://images.financialmodelingprep.com/symbo...,2006-08-17,True,False,False,False,False
89439,T1EU.F,40.2440,0.000000e+00,0.012456,0.00,40.244-42.595,-0.0280,-0.06950,2.950000e+02,31.0,Invesco US Treasury Bond 0-1 Year UCITS ETF,EUR,NaN,IE00BLCH1X54,NaN,Frankfurt Stock Exchange,FSX,Asset Management - Bonds,NaN,NaN,NaN,Financial Services,IE,NaN,NaN,NaN,NaN,NaN,NaN,https://images.fin

# FOREX

In [12]:
import os
import time
import random
import requests
import pandas as pd
import io
from dotenv import load_dotenv, find_dotenv

# Initialize (assuming .env is already loaded in your session, but safe to keep)
load_dotenv(find_dotenv())
api_key = os.getenv("FMP_API_KEY")

def robust_fmp_fetch(url, params, max_retries=5):
    """
    Generalized fetcher with exponential backoff and 
    flexible CSV/JSON handling.
    """
    for n in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            
            if response.status_code == 200:
                text = response.text.strip()
                if not text:
                    return pd.DataFrame()

                # Determine if response is JSON or CSV
                content_type = response.headers.get('Content-Type', '').lower()
                if 'csv' in content_type or not text.startswith('['):
                    return pd.read_csv(io.StringIO(text))
                else:
                    return pd.DataFrame(response.json())

            elif response.status_code == 429 or 500 <= response.status_code < 600:
                wait = (2 ** (n + 1)) + random.uniform(0, 1)
                print(f"Retry {n+1}/{max_retries} due to {response.status_code}. Waiting {wait:.2f}s...")
                time.sleep(wait)
            else:
                print(f"Permanent Error {response.status_code}: {text[:100]}")
                return None
                
        except Exception as e:
            wait = (2 ** (n + 1)) + random.uniform(0, 1)
            time.sleep(wait)
    return None

# --- Step 1: Get the Forex Symbol List ---
print("Fetching master Forex list...")
forex_list_url = "https://financialmodelingprep.com/stable/forex-list"
df_forex_list = robust_fmp_fetch(forex_list_url, {"apikey": api_key})

if df_forex_list is not None:
    df_forex_list.to_csv("data/forex-list.csv", index=False)
    print(f"Saved {len(df_forex_list)} symbols to data/forex-list.csv")

# --- Step 2: Get Real-Time Batch Quotes ---
print("\nFetching batch Forex quotes...")
# 'short=True' gives a lightweight response as per your requirement
quotes_url = "https://financialmodelingprep.com/stable/batch-forex-quotes"
df_forex_quotes = robust_fmp_fetch(quotes_url, {"apikey": api_key, "short": "true"})

if df_forex_quotes is not None:
    df_forex_quotes.to_csv("data/forex-quotes.csv", index=False)
    print(f"Saved {len(df_forex_quotes)} quotes to data/forex-quotes.csv")
    
    # Display top results
    print("\nSample Quotes:")
    print(df_forex_quotes.head())

Fetching master Forex list...
Saved 1545 symbols to data/forex-list.csv

Fetching batch Forex quotes...
Saved 1545 quotes to data/forex-quotes.csv

Sample Quotes:
   symbol    price    change  volume
0  AEDAUD  0.37978  0.003270     0.0
1  AEDBHD  0.10238 -0.000185    91.0
2  AEDCAD  0.37354  0.000430     0.0
3  AEDCHF  0.21415  0.001307     0.0
4  AEDDKK  1.78509 -0.085484    91.0


In [15]:
import os
import pandas as pd
from datetime import datetime

# 1. Ensure the output directory exists
os.makedirs("data/forex", exist_ok=True)

# 2. Load the master list
list_path = "data/forex-list.csv"
if os.path.exists(list_path):
    df_list = pd.read_csv(list_path)
else:
    print(f"Error: {list_path} not found.")
    df_list = None

# 3. Fetch Real-Time Batch Quotes
print("\nFetching batch Forex quotes...")
quotes_url = "https://financialmodelingprep.com/stable/batch-forex-quotes"
# Assuming robust_fmp_fetch is defined in your notebook from the previous step
df_forex_quotes = robust_fmp_fetch(quotes_url, {"apikey": api_key, "short": "true"})

if df_list is not None and df_forex_quotes is not None:
    # 4. Perform the Left Join on 'symbol'
    df_merged = pd.merge(df_list, df_forex_quotes, on='symbol', how='left')

    # 5. Generate the timestamped filename
    # Using your specific format: forex-YYYY-MM-DD-HH-MM.csv
    timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M")
    filename = f"data/forex/forex-{timestamp}.csv"

    # 6. Save the file
    df_merged.to_csv(filename, index=False)
    
    print(f"Successfully joined data.")
    print(f"File saved as: {filename}")
    print(f"Total rows: {len(df_merged)}")
    
    # 7. Preview - Fixed column names to match your CSV structure
    print("\nMerged Data Preview:")
    # Using 'fromName' and 'toName' since 'name' doesn't exist in your list CSV
    cols_to_show = ['symbol', 'fromName', 'toName', 'price', 'change']
    # We use intersection to avoid errors if some columns are missing
    existing_cols = [c for c in cols_to_show if c in df_merged.columns]
    print(df_merged[existing_cols].head())
else:
    print("Merge failed: Missing data from API or local CSV.")


Fetching batch Forex quotes...
Successfully joined data.
File saved as: data/forex/forex-2026-05-15-10-27.csv
Total rows: 1545

Merged Data Preview:
   symbol             fromName              toName        price      change
0  ARSMXN       Argentine Peso        Mexican Peso     0.012414    0.000063
1  TNDZAR       Tunisian Dinar  South African Rand     5.930800   -0.006999
2  XAGRUB  Silver (troy ounce)       Russian Ruble  2661.272510 -259.087490
3  ILSNOK   Israeli New Shekel     Norwegian Krone     2.860860   -0.053667
4  COPCLP       Colombian Peso        Chilean Peso     0.238700    0.001400


In [17]:
df_merged

,symbol,fromCurrency,toCurrency,fromName,toName,price,change,volume
0,ARSMXN,ARS,MXN,Argentine Peso,Mexican Peso,0.012414,0.000063,1362435.0
1,TNDZAR,TND,ZAR,Tunisian Dinar,South African Rand,5.930800,-0.006999,7.0
2,XAGRUB,XAG,RUB,Silver (troy ounce),Russian Ruble,2661.272510,-259.087490,1254.0
3,ILSNOK,ILS,NOK,Israeli New Shekel,Norwegian Krone,2.860860,-0.053667,154608.0
4,COPCLP,COP,CLP,Colombian Peso,Chilean Peso,0.238700,0.001400,304.0
...,...,...,...,...,...,...,...,...
1540,BNDEUR,BND,EUR,Brunei Dollar,Euro,0.672310,0.000136,0.0
1541,NADZAR,NAD,ZAR,Namibian Dollar,South African Rand,1.000000,0.000000,6893.0
1542,KESGBP,KES,GBP,Kenyan Shilling,British Pound Sterling,0.005800,0.000030,0.0
1543,ZARXAU,ZAR,XAU,South African Rand,Gold (troy ounce),0.000016,-0.000002,1256.0


# Company Profiles and USD Mkt Cap

In [18]:
full_df['currency'].unique()

<StringArray>
['USD', 'CAD', 'EUR', 'RUB',   nan, 'MXN', 'BRL', 'GBp', 'COP', 'CHF', 'SEK',
 'TWD', 'ARS', 'CLP', 'GBP', 'NOK', 'ISK', 'TRY', 'ZAc', 'HUF', 'PLN', 'DKK',
 'CZK', 'AED', 'ILA', 'INR', 'JPY', 'THB', 'SGD', 'MYR', 'HKD', 'CNY', 'VND',
 'KRW', 'AUD', 'NZD', 'ZAR', 'SAR', 'EGP', 'QAR', 'KWF', 'IDR', 'ILS', 'NGN',
 'RON', 'KZT', 'LBP', 'KES']
Length: 48, dtype: str

In [59]:
# Drop the NANs in 'currency' column

full_df = full_df.dropna(subset=['currency'])

In [60]:
import numpy as np

# 1. Create a lookup table for Currency -> USD conversion
# We want the price where fromCurrency is the local coin and toCurrency is USD
usd_rates = df_merged[df_merged['toCurrency'] == 'USD'][['fromCurrency', 'price']].copy()
usd_rates.columns = ['currency', 'fx_rate']

# 2. Add USD to the lookup table (rate is 1.0)
usd_rate_map = dict(zip(usd_rates['currency'], usd_rates['fx_rate']))
usd_rate_map['USD'] = 1.0

# 3. Handle the "GBp" (Pence) vs "GBP" (Pound) issue
# If the currency is GBp, it uses the GBP rate but needs to be divided by 100
if 'GBP' in usd_rate_map:
    usd_rate_map['GBp'] = usd_rate_map['GBP'] / 100.0

# 4. Map the FX rates to your full_df
full_df['fx_rate'] = full_df['currency'].map(usd_rate_map)

# 5. Calculate Market Cap in USD
full_df['marketCapUSD'] = full_df['marketCap'] * full_df['fx_rate']

# 6. Preview the results for foreign firms
foreign_firms = full_df[full_df['currency'] != 'USD'].copy()

print(f"Calculated USD Market Cap for {len(full_df) - full_df['marketCapUSD'].isna().sum()} firms.")
print("\nSample Conversion (Foreign Firms):")
print(foreign_firms[['symbol', 'companyName', 'currency', 'marketCap', 'fx_rate', 'marketCapUSD']].head())

Calculated USD Market Cap for 88560 firms.

Sample Conversion (Foreign Firms):
        symbol                                        companyName currency  \
55   FCR-UN.TO         First Capital Real Estate Investment Trust      CAD   
73      DUL.DE                      Alnylam Pharmaceuticals, Inc.      EUR   
204    KZMS.ME  The Open Joint Stock Company Krasnokamsk Metal...      RUB   
249     ARX.TO                                 ARC Resources Ltd.      CAD   
255     MFC.TO                     Manulife Financial Corporation      CAD   

            marketCap  fx_rate      marketCapUSD  
55   4,928,087,675.00     0.73  3,585,676,592.33  
73  33,290,263,545.00     1.16 38,731,890,024.07  
204    199,955,760.00     0.01      2,736,794.49  
249 17,559,195,560.00     0.73 12,776,070,689.46  
255 86,620,388,144.00     0.73 63,024,994,413.57  


In [61]:
full_df = full_df[full_df['marketCapUSD'] > 0].copy()

In [62]:
full_df = full_df[full_df['isActivelyTrading'] == True].copy()

In [39]:
full_df

,symbol,price,marketCap,beta,lastDividend,range,change,changePercentage,volume,averageVolume,companyName,currency,cik,isin,cusip,exchangeFullName,exchange,industry,website,description,ceo,sector,country,fullTimeEmployees,phone,address,city,state,zip,image,ipoDate,defaultImage,isEtf,isActivelyTrading,isAdr,isFund,fx_rate,marketCapUSD
0,NVDA,228.1900,5.545017e+12,2.244000,0.04,129.16-236.54,-7.550000e+00,-3.202680,9.265985e+07,1.702981e+08,NVIDIA Corporation,USD,1045810.0,US67066G1040,67066G104,NASDAQ Global Select,NASDAQ,Semiconductors,https://www.nvidia.com,"NVIDIA Corporation provides graphics, and comp...",Jen-Hsun Huang,Technology,US,36000.0,408 486 2000,2788 San Tomas Expressway,Santa Clara,CA,95051,https://images.financialmodelingprep.com/symbo...,1999-01-22,False,False,True,False,False,1.0,5.545017e+12
1,KVYO,14.6300,4.378464e+09,0.791000,0.00,13.53-36.76,2.900000e-01,2.022320,2.833205e+06,5.295504e+06,"Klaviyo, Inc.",USD,1835830.0,US49845K1016,49845K101,New York Stock Exchange,NYSE,Software - Infrastructure,https://www.klaviyo.com,"Klaviyo, Inc., a technology company that provi...",Andrew Bialecki,Technology,US,2316.0,(617) 213-1788,Floor 6,Boston,MA,02111,https://images.financialmodelingprep.com/symbo...,2023-09-20,False,False,True,False,False,1.0,4.378464e+09
2,SATS,137.3499,3.958629e+10,0.960000,0.00,14.9-139.54,2.239900e+00,1.657830,2.465327e+06,6.007778e+06,EchoStar Corporation,USD,1415404.0,US2787681061,278768106,NASDAQ Global Select,NASDAQ,Communication Equipment,https://www.echostar.com,"EchoStar Corporation, together with its subsid...",Hamid Akhavan-Malayeri,Technology,US,13700.0,303 706 4000,100 Inverness Terrace East,Englewood,CO,80112-5308,https://images.financialmodelingprep.com/symbo...,2008-01-02,False,False,True,False,False,1.0,3.958629e+10
3,SHC,15.4149,4.395821e+09,1.823000,0.00,10.795-19.85,-7.510000e-02,-0.484830,2.577312e+06,3.416172e+06,Sotera Health Company,USD,1822479.0,US83601L1026,83601L102,NASDAQ Global Select,NASDAQ,Medical - Diagnostics & Research,https://soterahealth.com,"Sotera Health Company provides sterilization, ...",Michael Petras Jr.,Healthcare,US,3000.0,440 262 1410,9100 South Hills Boulevard,Broadview Heights,OH,44147,https://images.financialmodelingprep.com/symbo...,2020-11-20,False,False,True,False,False,1.0,4.395821e+09
4,STRC,99.2888,2.949599e+10,3.595000,11.50,90.52-100.418,-7.112000e-01,-0.711200,3.666699e+06,2.985974e+06,MicroStrategy Incorporated Variable Rate Serie...,USD,1050446.0,US5949728530,594972408,NASDAQ Global Select,NASDAQ,Software - Application,http://www.strategy.com,"MicroStrategy, Inc. engages in the provision o...",Phong Q. Le,Technology,US,1534.0,703 848 8600,1850 Towers Crescent Plaza,Vienna,VA,22182,https://images.financialmodelingprep.com/symbo...,2025-07-30,True,False,True,False,False,1.0,2.949599e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86483,ALPC,20.0000,8.092220e+08,0.000000,0.00,20-20,0.000000e+00,0.000000,1.000000e+02,4.008032e+01,Alpha Investment Inc.,USD,1616736.0,NaN,NaN,Other OTC,OTC,Financial - Credit Services,https://www.alphainvestmentinc.com,Alpha Investment Inc. focuses on the provision...,Todd Buxton,Financial Services,US,NaN,305-704-3294,200 East Campus View Boulevard,Columbus,OH,43235,https://images.financialmodelingprep.com/symbo...,2019-06-28,False,False,True,False,False,1.0,8.092220e+08
86602,COPGF,0.0800,8.245360e+06,1.063000,0.00,0.08-0.08,-2.000000e-09,-0.000003,6.250000e+02,2.510000e+02,Olympio Metals Limited,USD,NaN,AU0000221632,NaN,Other OTC,OTC,Industrial Materials,https://www.olympiometals.com.au,Olympio Metals Limited engages in gold explora...,John Gerard Delaney,Basic Materials,AU,NaN,61 8 9322 9093,25 Richardson St,West Perth,WA,6005,https://images.financialmodelingprep.com/symbo...,2022-11-11,True,False,True,False,False,1.0,8.245360e+06
87320,GCRCF,0.0192,4.817720e+05,0.565000,0.00,0.0192-0.2252,-2.06000

In [63]:
import pandas as pd

# Set pandas to show numbers in a readable format (not scientific notation)
pd.options.display.float_format = '{:,.2f}'.format

# Grouping by country
country_stats = full_df.groupby('country').agg(
    total_market_cap_usd=('marketCapUSD', 'sum'),
    company_count=('symbol', 'count')
).sort_values(by='total_market_cap_usd', ascending=False)

# Convert to Billions for easier reading
country_stats['market_cap_billions'] = country_stats['total_market_cap_usd'] / 1e9

print("Top 10 Countries by Total Market Cap:")
print(country_stats[['market_cap_billions', 'company_count']].head(10))

Top 10 Countries by Total Market Cap:
         market_cap_billions  company_count
country                                    
US                340,093.18          18493
CN                 42,345.14           6159
JP                 24,106.80           4538
IE                 23,963.36           3207
KR                 16,850.25           1631
CA                 14,283.00           4910
FR                 12,295.23           1016
GB                 11,702.75           1931
IN                 10,840.24           6325
CH                 10,348.12            599


In [ ]:
full_df['sector'] = full_df['sector'].replace('onsumer Cyclical', 'Consumer Cyclical')

sector_stats = full_df.groupby('sector').agg(
    total_market_cap_usd=('marketCapUSD', 'sum'),
    avg_beta=('beta', 'mean'),
    company_count=('symbol', 'count')
).sort_values(by='total_market_cap_usd', ascending=False)

sector_stats['market_cap_trillions'] = sector_stats['total_market_cap_usd'] / 1e12

print("\nMarket Cap and Risk (Beta) by Sector:")
print(sector_stats[['market_cap_trillions', 'avg_beta', 'company_count']])


Market Cap and Risk (Beta) by Sector:
                        market_cap_trillions  avg_beta  company_count
sector                                                               
Financial Services                    196.00    -10.10          21605
Technology                            131.36      5.43           6669
Communication Services                 62.02     -0.10           2010
Industrials                            44.82     -4.77           9152
Consumer Cyclical                      44.50    -13.55           6069
Healthcare                             29.66   -258.63           5256
Consumer Defensive                     22.71     -0.16           3004
Energy                                 22.51     -2.81           2004
Basic Materials                        19.00     -1.40           6948
Utilities                              10.91      0.26           1203
Real Estate                             8.04      0.75           2691


In [68]:
top_giants = full_df.nlargest(10, 'marketCapUSD')[['symbol', 'companyName', 'country', 'sector', 'marketCapUSD']]
top_giants['market_cap_trillions'] = top_giants['marketCapUSD'] / 1e12

print("\nTop 10 Global Companies by Market Cap (USD Trillions):")
print(top_giants[['symbol', 'companyName', 'country', 'market_cap_trillions']])


Top 10 Global Companies by Market Cap (USD Trillions):
        symbol         companyName country  market_cap_trillions
17462   NVD.DE  NVIDIA Corporation      US                  5.55
0         NVDA  NVIDIA Corporation      US                  5.55
11553    NVD.F  NVIDIA Corporation      US                  5.54
7811   NVDA.NE  NVIDIA Corporation      US                  5.28
17293  ABEA.DE       Alphabet Inc.      US                  4.81
11499   ABEA.F       Alphabet Inc.      US                  4.81
203      GOOGL       Alphabet Inc.      US                  4.80
17032  ABEC.DE       Alphabet Inc.      US                  4.77
123       GOOG       Alphabet Inc.      US                  4.75
13492   ABEC.F       Alphabet Inc.      US                  4.75


In [69]:
full_df.to_csv('data/company-profiles/company-profiles-bulk-usd-mktcap-cleaned.csv', index=False)

# Earnings

In [70]:
import os
import time
import random
import requests
import pandas as pd
import io
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

# 1. Setup
load_dotenv(find_dotenv())
api_key = os.getenv("FMP_API_KEY")
os.makedirs("data/earnings", exist_ok=True)

def fetch_with_backoff(url, params, max_retries=6):
    """
    Performs GET with exponential backoff: 2^n + jitter.
    Handles both JSON and CSV bulk responses.
    """
    for n in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            
            # Success
            if response.status_code == 200:
                text = response.text.strip()
                if not text:
                    return pd.DataFrame() # Return empty DF if year has no data
                
                # Detect format: CSV vs JSON
                if text.startswith('[') or text.startswith('{'):
                    return pd.DataFrame(response.json())
                else:
                    return pd.read_csv(io.StringIO(text))

            # Retryable Errors: 429 (Rate Limit) or 5xx (Server)
            elif response.status_code == 429 or 500 <= response.status_code < 600:
                wait_time = (2 ** (n + 1)) + random.uniform(0, 1)
                print(f"      [Retry {n+1}/{max_retries}] Status {response.status_code}. Waiting {wait_time:.2f}s...")
                time.sleep(wait_time)
            
            else:
                print(f"      [Error] Permanent status {response.status_code}. Skipping.")
                return None

        except Exception as e:
            wait_time = (2 ** (n + 1)) + random.uniform(0, 1)
            print(f"      [Connection Error] {e}. Retrying in {wait_time:.2f}s...")
            time.sleep(wait_time)
            
    return None

# 2. Main Loop: From 2026 back to 1990
current_year = 2026
start_year = 1990
all_yearly_dfs = []

print(f"--- Starting Reverse Bulk Download ({current_year} -> {start_year}) ---")

for year in range(current_year, start_year - 1, -1):
    print(f"Fetching Year: {year}...")
    url = "https://financialmodelingprep.com/stable/earnings-surprises-bulk"
    params = {"year": str(year), "apikey": api_key}
    
    df_year = fetch_with_backoff(url, params)
    
    if df_year is not None and not df_year.empty:
        # Add year column for easy filtering later
        df_year['year_group'] = year
        
        # Save individual year to prevent total data loss if script stops
        year_path = f"data/earnings/surprises_{year}.csv"
        df_year.to_csv(year_path, index=False)
        
        all_yearly_dfs.append(df_year)
        print(f"   Successfully saved {len(df_year)} records for {year}.")
    else:
        # If we get multiple empty years in a row, we've likely hit the end of history
        print(f"   No data found for {year}. Moving back...")

    # Gentle pause between years to prevent aggressive rate limiting
    time.sleep(0.5)

# 3. Final Consolidation
if all_yearly_dfs:
    master_df = pd.concat(all_yearly_dfs, ignore_index=True)
    master_df.to_csv("data/earnings/earnings_surprises_full.csv", index=False)
    print("\n" + "="*40)
    print(f"COMPLETE: {len(master_df)} total records consolidated.")
    print("Final File: data/earnings/earnings_surprises_full.csv")
    print("="*40)

--- Starting Reverse Bulk Download (2026 -> 1990) ---
Fetching Year: 2026...
   Successfully saved 30279 records for 2026.
Fetching Year: 2025...
      [Retry 1/5] Status 429. Waiting 2.03s...
      [Retry 2/5] Status 429. Waiting 4.33s...
      [Retry 3/5] Status 429. Waiting 8.61s...
   Successfully saved 52752 records for 2025.
Fetching Year: 2024...
      [Retry 1/5] Status 429. Waiting 2.67s...
      [Retry 2/5] Status 429. Waiting 4.89s...
   Successfully saved 61488 records for 2024.
Fetching Year: 2023...
      [Retry 1/5] Status 429. Waiting 2.36s...
      [Retry 2/5] Status 429. Waiting 4.86s...
   Successfully saved 60768 records for 2023.
Fetching Year: 2022...
      [Retry 1/5] Status 429. Waiting 2.35s...
      [Retry 2/5] Status 429. Waiting 4.83s...
      [Retry 3/5] Status 429. Waiting 8.11s...
   Successfully saved 60054 records for 2022.
Fetching Year: 2021...
      [Retry 1/5] Status 429. Waiting 2.02s...
      [Retry 2/5] Status 429. Waiting 4.32s...
      [Retry 3

In [74]:
master_df.sort_values(by=[ 'symbol', 'date'], inplace=True)

# First Profitability

In [76]:
import pandas as pd

# 1. Ensure data types are correct (API data is often strings)
master_df['date'] = pd.to_datetime(master_df['date'])
master_df['epsActual'] = pd.to_numeric(master_df['epsActual'], errors='coerce')

# 2. Filter for rows where EPS is strictly positive
positive_eps_df = master_df[master_df['epsActual'] > 0.01].copy()

# 3. Since it's already sorted by date, we just drop duplicates of the symbol
# keeping only the 'first' instance (the earliest date)
first_positive_eps = positive_eps_df.drop_duplicates(subset=['symbol'], keep='first')

# 4. Clean up and sort for a nice view
first_positive_eps = first_positive_eps[['symbol', 'date', 'epsActual']].sort_values('date')

print(f"Identified first positive EPS dates for {len(first_positive_eps)} unique symbols.")
print("\nSample Output (Earliest Profitability Milestones):")
print(first_positive_eps.head(10))

# 5. Optional: Save this to your data folder
# first_positive_eps.to_csv("data/earnings/first_profitability_milestones.csv", index=False)

Identified first positive EPS dates for 24672 unique symbols.

Sample Output (Earliest Profitability Milestones):
       symbol       date  epsActual
715232    SVU 1990-02-27       2.06
715231   ESND 1990-02-27       0.03
715228   FFKT 1990-03-30       0.29
715223   LNCE 1990-03-30       0.34
715227    AET 1990-03-30       0.40
715226    CAA 1990-03-30       1.70
715224    EGN 1990-03-30       0.28
715225  GNCMA 1990-03-30       0.03
715208   ESIO 1990-08-30       0.04
715200    TWX 1990-09-29       0.03


In [79]:
first_positive_eps.tail(10)

,symbol,date,epsActual
514,TOPPY,2026-05-14,0.08
444,VSNT,2026-05-14,1.99
272,PIIIW,2026-05-14,0.32
692,ONDS,2026-05-14,0.81
352,PIII,2026-05-14,0.32
705,AYA,2026-05-14,0.33
459,5123.KL,2026-05-14,0.02
731,UMAC,2026-05-14,0.21
11,TAWNF,2026-05-15,0.01
42,7327.T,2026-05-15,31.23


# Prices

In [1]:
import pyarrow.parquet as pq

# Read just the metadata/schema
parquet_file = pq.ParquetFile('/home/vince/repos/techno-quantamental-analyzer/notebooks/fmp-full-download/data/prices/N/NSGAX.parquet')
print(parquet_file.schema)

required group field_id=-1 schema {
  optional binary field_id=-1 symbol (String);
  optional int64 field_id=-1 date (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional double field_id=-1 open;
  optional double field_id=-1 high;
  optional double field_id=-1 low;
  optional double field_id=-1 close;
  optional int64 field_id=-1 volume;
  optional int64 field_id=-1 change;
  optional int64 field_id=-1 changePercent;
  optional double field_id=-1 vwap;
}



In [4]:
import pandas as pd
df = pd.read_parquet('/home/vince/repos/techno-quantamental-analyzer/notebooks/fmp-full-download/data/prices/W/WAB.parquet')
df

,symbol,date,open,high,low,close,volume,change,changePercent,vwap
0,WAB,1995-06-16,7.50,7.75,7.440,7.500,6212400,0.0000,0.00000,7.5475
1,WAB,1995-06-19,7.56,7.63,7.060,7.190,1528200,-0.3750,-4.89000,7.3600
2,WAB,1995-06-20,7.25,7.31,7.190,7.190,863000,-0.0625,-0.82759,7.2350
3,WAB,1995-06-21,7.25,7.25,7.000,7.060,629000,-0.1875,-2.62000,7.1400
4,WAB,1995-06-22,7.06,7.56,7.000,7.310,546200,0.2500,3.54000,7.2325
...,...,...,...,...,...,...,...,...,...,...
7775,WAB,2026-05-11,265.54,268.32,263.010,268.130,772534,2.5900,0.97537,266.2500
7776,WAB,2026-05-12,266.99,270.60,262.810,269.020,782718,2.0300,0.76033,267.3550
7777,WAB,2026-05-13,269.93,271.36,264.710,264.780,1049255,-5.1500,-1.91000,267.6950
7778,WAB,2026-05-14,266.20,269.44,264.940,269.430,1791755,3.2300,1.21000,267.5025


In [3]:
df

,symbol,date,open,high,low,close,volume,change,changePercent,vwap
0,NSGAX,1999-07-30,13.88,13.88,13.88,13.88,0,0,0,13.88
1,NSGAX,1999-08-02,13.88,13.88,13.88,13.88,0,0,0,13.88
2,NSGAX,1999-08-03,13.79,13.79,13.79,13.79,0,0,0,13.79
3,NSGAX,1999-08-04,13.54,13.54,13.54,13.54,0,0,0,13.54
4,NSGAX,1999-08-05,13.61,13.61,13.61,13.61,0,0,0,13.61
...,...,...,...,...,...,...,...,...,...,...
6735,NSGAX,2026-05-11,25.51,25.51,25.51,25.51,0,0,0,25.51
6736,NSGAX,2026-05-12,25.56,25.56,25.56,25.56,0,0,0,25.56
6737,NSGAX,2026-05-13,25.74,25.74,25.74,25.74,0,0,0,25.74
6738,NSGAX,2026-05-14,25.99,25.99,25.99,25.99,0,0,0,25.99
